# Netflix Titles Data Cleaning
This notebook cleans the `netflix_titles.csv` dataset.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 3.9 MB


In [4]:
df.describe(include='all')

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
count,8807,8807,8807,6173,7982,7976,8797,8807.000000,8803,8804,8807,8807
unique,8807,2,8807,4528,7692,748,1767,NaN,17,220,514,8775
top,s1,Movie,Dick Johnson Is Dead,Rajiv Chilaka,David Attenborough,United States,"January 1, 2020",NaN,TV-MA,1 Season,"Dramas, International Movies","Paranormal activity at a lush, abandoned prope..."
freq,1,6131,1,19,19,2818,109,NaN,3207,1793,362,4
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2014.180198,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.819312,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1925.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2013.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2019.000000,NaN,NaN,NaN,NaN


In [5]:
missing_counts = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_counts, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0]

,Missing Count,Missing %
director,2634,29.908028
cast,825,9.367549
country,831,9.435676
date_added,10,0.113546
rating,4,0.045418
duration,3,0.034064


## Cleaning Steps

In [6]:
df_clean = df.copy()

### 1. Strip whitespace from all text columns first
Việc strip whitespace TRƯỚC sẽ giúp các bước sau (parse date, split string) hoạt động chính xác.

In [7]:
text_cols = ['title', 'director', 'country', 'rating', 'listed_in', 'description', 'date_added', 'cast', 'duration']
for col in text_cols:
    df_clean[col] = df_clean[col].str.strip()

# Kiem tra khong con whitespace
whitespace_check = sum(df_clean[col].str.contains(r'^\s|\s$', na=False).sum() for col in text_cols)
print(f'Rows con leading/trailing whitespace: {whitespace_check}')
df_clean[text_cols].head()

Rows con leading/trailing whitespace: 0


,title,director,country,rating,listed_in,description,date_added,cast,duration
0,Dick Johnson Is Dead,Kirsten Johnson,United States,PG-13,Documentaries,"As her father nears the end of his life, filmm...","September 25, 2021",NaN,90 min
1,Blood & Water,NaN,South Africa,TV-MA,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...","September 24, 2021","Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",2 Seasons
2,Ganglands,Julien Leclercq,NaN,TV-MA,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,"September 24, 2021","Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",1 Season
3,Jailbirds New Orleans,NaN,NaN,TV-MA,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...","September 24, 2021",NaN,1 Season
4,Kota Factory,NaN,India,TV-MA,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,"September 24, 2021","Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",2 Seasons


### 2. Convert `date_added` to datetime
Strip whitespace trước khi parse để tránh lỗi format mismatch.

In [8]:
# Parse date_added - dung format chuan, dam bao da strip whitespace
df_clean['date_added'] = pd.to_datetime(df_clean['date_added'], format='%B %d, %Y', errors='coerce')

print(f'Số NaT trong date_added: {df_clean["date_added"].isna().sum()}')
print(f'(Trong đó {df["date_added"].isna().sum()} là NaN gốc, còn lại {df_clean["date_added"].isna().sum() - df["date_added"].isna().sum()} là parse lỗi)')
df_clean[['date_added']].head()

Số NaT trong date_added: 10
(Trong đó 10 là NaN gốc, còn lại 0 là parse lỗi)


,date_added
0,2021-09-25
1,2021-09-24
2,2021-09-24
3,2021-09-24
4,2021-09-24


### 3. Parse `duration` into numeric columns

In [9]:
df_clean['duration_min'] = df_clean['duration'].str.extract(r'(\d+)\s*min').astype(float)
df_clean['duration_seasons'] = df_clean['duration'].str.extract(r'(\d+)\s*Season').astype(float)

print(f'duration_min NaN: {df_clean["duration_min"].isna().sum()}')
print(f'duration_seasons NaN: {df_clean["duration_seasons"].isna().sum()}')
print(f'duration gốc NaN: {df_clean["duration"].isna().sum()}')
df_clean[['duration', 'duration_min', 'duration_seasons']].head(10)

duration_min NaN: 2679
duration_seasons NaN: 6131
duration gốc NaN: 3


,duration,duration_min,duration_seasons
0,90 min,90.0,NaN
1,2 Seasons,NaN,2.0
2,1 Season,NaN,1.0
3,1 Season,NaN,1.0
4,2 Seasons,NaN,2.0
5,1 Season,NaN,1.0
6,91 min,91.0,NaN
7,125 min,125.0,NaN
8,9 Seasons,NaN,9.0
9,104 min,104.0,NaN


### 4. Fix cột `rating` - chuyển các giá trị duration bị nhầm sang duration 
Một số rows có giá trị duration (vd: '66 min', '74 min', '84 min') bị ghi nhầm vào cột rating.
Chỉ chuyển nếu duration_min chưa được parse từ duration gốc (tránh ghi đè làm hỏng dữ liệu).

In [10]:
# Phat hien cac gia tri rating la duration thay vi rating that
rating_is_duration = df_clean['rating'].str.match(r'^\d+\s*min$', na=False)
print(f'So rating bi nham duration: {rating_is_duration.sum()}')
print(df_clean[rating_is_duration][['show_id', 'title', 'type', 'rating', 'duration']].to_string())

# Chi chuyen neu duration_min chua duoc parse (tranh ghi de du lieu dung)
for idx in df_clean[rating_is_duration].index:
    dur_val = df_clean.loc[idx, 'rating']
    if pd.isna(df_clean.loc[idx, 'duration_min']):
        df_clean.loc[idx, 'duration_min'] = pd.to_numeric(dur_val.replace(' min', ''), errors='coerce')
    df_clean.loc[idx, 'rating'] = 'Unknown'

print(f'\nSau khi fix, cac rating con lai:')
print(df_clean['rating'].unique())

So rating bi nham duration: 3
     show_id                                 title   type  rating duration
5541   s5542                       Louis C.K. 2017  Movie  74 min      NaN
5794   s5795                 Louis C.K.: Hilarious  Movie  84 min      NaN
5813   s5814  Louis C.K.: Live at the Comedy Store  Movie  66 min      NaN

Sau khi fix, cac rating con lai:
<ArrowStringArray>
[   'PG-13',    'TV-MA',       'PG',    'TV-14',    'TV-PG',     'TV-Y',
    'TV-Y7',        'R',     'TV-G',        'G',    'NC-17',  'Unknown',
       'NR',        nan, 'TV-Y7-FV',       'UR']
Length: 16, dtype: str


### 5. Fill missing values appropriately

In [11]:
# Fill categorical missing values with 'Unknown'
df_clean['director'] = df_clean['director'].fillna('Unknown')
df_clean['cast'] = df_clean['cast'].fillna('Unknown')
df_clean['country'] = df_clean['country'].fillna('Unknown')
df_clean['rating'] = df_clean['rating'].fillna('Unknown')
df_clean['duration'] = df_clean['duration'].fillna('Unknown')

df_clean[['director', 'cast', 'country', 'rating', 'duration']].isnull().sum()

director    0
cast        0
country     0
rating      0
duration    0
dtype: int64

### 6. Split comma-separated columns into lists

In [12]:
df_clean['cast_list'] = df_clean['cast'].apply(lambda x: [c.strip() for c in x.split(',')] if x != 'Unknown' else [])
df_clean['listed_in_list'] = df_clean['listed_in'].apply(lambda x: [c.strip() for c in x.split(',')])
df_clean['country_list'] = df_clean['country'].apply(lambda x: [c.strip() for c in x.split(',')] if x != 'Unknown' else [])

df_clean[['cast', 'cast_list']].head(3)

,cast,cast_list
0,Unknown,[]
1,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...","[Ama Qamata, Khosi Ngema, Gail Mabalane, Thaba..."
2,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...","[Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nab..."


### 7. Clean country_list - filter out empty strings
Fix: một số country bắt đầu bằng dấu phẩy (vd: ', South Korea') tạo phần tử rỗng trong country_list.
Loại bỏ phần tử rỗng để country_list sạch, giống cast_list (giữ nguyên danh sách các quốc gia).

In [13]:
# Chuan hoa country_list: loai bo phan tu rong (vd: ', South Korea' -> ['', 'South Korea'])
df_clean['country_list'] = df_clean['country_list'].apply(lambda lst: [c for c in lst if c])

print(f'So country_list rong: {df_clean["country_list"].apply(lambda x: len(x) == 0).sum()}')
print(f'So country_list co 1 quoc gia: {df_clean["country_list"].apply(lambda x: len(x) == 1).sum()}')
print(f'So country_list co nhieu quoc gia: {df_clean["country_list"].apply(lambda x: len(x) > 1).sum()}')
df_clean[['country', 'country_list']].head(10)

So country_list rong: 831
So country_list co 1 quoc gia: 6661
So country_list co nhieu quoc gia: 1315


,country,country_list
0,United States,[United States]
1,South Africa,[South Africa]
2,Unknown,[]
3,Unknown,[]
4,India,[India]
5,Unknown,[]
6,Unknown,[]
7,"United States, Ghana, Burkina Faso, United Kin...","[United States, Ghana, Burkina Faso, United Ki..."
8,United Kingdom,[United Kingdom]
9,United States,[United States]


### 8. Remove duplicate rows

In [14]:
print(f'Rows before dedup: {len(df_clean)}')
df_clean = df_clean.drop_duplicates(subset=['title', 'type', 'release_year', 'director'], keep='first')
print(f'Rows after dedup: {len(df_clean)}')

Rows before dedup: 8807
Rows after dedup: 8806


### 9. Final check

In [15]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 8806 entries, 0 to 8806
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   show_id           8806 non-null   str           
 1   type              8806 non-null   str           
 2   title             8806 non-null   str           
 3   director          8806 non-null   str           
 4   cast              8806 non-null   str           
 5   country           8806 non-null   str           
 6   date_added        8796 non-null   datetime64[us]
 7   release_year      8806 non-null   int64         
 8   rating            8806 non-null   str           
 9   duration          8806 non-null   str           
 10  listed_in         8806 non-null   str           
 11  description       8806 non-null   str           
 12  duration_min      6130 non-null   float64       
 13  duration_seasons  2676 non-null   float64       
 14  cast_list         8806 non-null   object

In [16]:
df_clean.head(5)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,duration_min,duration_seasons,cast_list,listed_in_list,country_list
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",90.0,NaN,[],[Documentaries],[United States]
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",NaN,2.0,"[Ama Qamata, Khosi Ngema, Gail Mabalane, Thaba...","[International TV Shows, TV Dramas, TV Mysteries]",[South Africa]
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,NaN,1.0,"[Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nab...","[Crime TV Shows, International TV Shows, TV Ac...",[]
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",NaN,1.0,[],"[Docuseries, Reality TV]",[]
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,NaN,2.0,"[Mayur More, Jitendra Kumar, Ranjan Raj, Alam ...","[International TV Shows, Romantic TV Shows, TV...",[India]


### 10. Validate cleaning quality

In [17]:
print('=== MISSING VALUES SAU CLEAN ===')
missing_after = df_clean.isnull().sum()
missing_after_pct = (df_clean.isnull().sum() / len(df_clean)) * 100
mv = pd.DataFrame({'Missing Count': missing_after, 'Missing %': missing_after_pct})
print(mv[mv['Missing Count'] > 0])

print()
print('=== KIEM TRA DATE_ADDED ===')
print(f'dtype: {df_clean["date_added"].dtype}')
print(f'NaT: {df_clean["date_added"].isna().sum()} (chi con NaN goc khong parse duoc)')

print()
print('=== KIEM TRA COUNTRY_LIST ===')
print(f'NaN: {df_clean["country_list"].isna().sum()}')
print(f'So dong co country_list rong: {df_clean["country_list"].apply(lambda x: len(x) == 0).sum()}')

print()
print('=== KIEM TRA RATING ===')
print(f'Unique ratings: {sorted(df_clean["rating"].unique())}')

print()
print('=== KIEM TRA DURATION ===')
print(f'duration_min NaN: {df_clean["duration_min"].isna().sum()}')
print(f'duration_seasons NaN: {df_clean["duration_seasons"].isna().sum()}')
print(f'duration Unknown: {(df_clean["duration"] == "Unknown").sum()}')

=== MISSING VALUES SAU CLEAN ===
                  Missing Count  Missing %
date_added                   10   0.113559
duration_min               2676  30.388372
duration_seasons           6130  69.611628

=== KIEM TRA DATE_ADDED ===
dtype: datetime64[us]
NaT: 10 (chi con NaN goc khong parse duoc)

=== KIEM TRA COUNTRY_LIST ===
NaN: 0
So dong co country_list rong: 831

=== KIEM TRA RATING ===
Unique ratings: ['G', 'NC-17', 'NR', 'PG', 'PG-13', 'R', 'TV-14', 'TV-G', 'TV-MA', 'TV-PG', 'TV-Y', 'TV-Y7', 'TV-Y7-FV', 'UR', 'Unknown']

=== KIEM TRA DURATION ===
duration_min NaN: 2676
duration_seasons NaN: 6130
duration Unknown: 3


### 11. Export cleaned data
Xuat ra CSV (dung cho hau het tools) va Parquet (giu nguyen kieu du lieu).

In [18]:
# Export CSV
df_clean.to_csv('netflix_titles_cleaned.csv', index=False)
print(f'Exported CSV: {len(df_clean)} rows, {len(df_clean.columns)} columns')

# Export Parquet (Parquet ho tro list type natively, pandas tu dong serialize list objects)
df_clean.to_parquet('netflix_titles_cleaned.parquet', index=False)
print(f'Exported Parquet: {len(df_clean)} rows, {len(df_clean.columns)} columns')

Exported CSV: 8806 rows, 17 columns
Exported Parquet: 8806 rows, 17 columns
